# <u> Monthly Gym Pledge Analytics </u>

- Author: Srikar Gunisetty
- Last Modified Date: 01/03/2026

**<u> Objective:</u>** This notebook analyzes a Google Sheets log of workouts with the following fields:

- Timestamp: when the form response was submitted
- Name: who submitted
- Workout date: the date the workout occurred
- Burnt >= 250 calories?: a boolean (Yes/No, True/False) indicating whether the workout meets the 250+ calories threshold

**<u>Primary goal:</u>**
- Determine monthly winners (>= 16 unique workout days in a month where Burnt >= 250 calories is true)

**<u>Secondary goals:</u>**
- Track consistency (including workouts under 250 calories)
- Day-of-week patterns
- Logging delay (timestamp vs workout date)
- Front-loading vs end-of-month cramming
- Streaks and inconsistency
- "Barely missed" (e.g., 14–15 qualifying days)
- Month-over-month trends and gamification awards

## <u> Setup & Data Import</u>

In [4]:
# libraries
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import gspread
from google.oauth2.service_account import Credentials

#set plot theme
sns.set_theme(style = "darkgrid", palette = "muted")

#set plot preferences
mpl.rcParams["figure.dpi"] = 150
plt.rcParams["font.family"] = "Consolas"
%matplotlib inline

import warnings
warnings.simplefilter(action = "ignore", category = FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [5]:
# 1) Your spreadsheet id is the long string in the URL
SPREADSHEET_ID = "17RADj_LH-Lj_lB8QFZyxjv8iKFerNZfNT-7wmtPNuzk"

# 2) This must match your tab name exactly (bottom of the Google Sheet)
WORKSHEET_NAME = "Form Responses"

# 3) Path to the JSON key you downloaded
SERVICE_ACCOUNT_JSON_PATH = "secrets/service_account.json"

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/drive.readonly",
]

def read_google_sheet_as_df(spreadsheet_id: str, worksheet_name: str, json_path: str) -> pd.DataFrame:
    creds = Credentials.from_service_account_file(json_path, scopes=SCOPES)
    gc = gspread.authorize(creds)

    sh = gc.open_by_key(spreadsheet_id)
    ws = sh.worksheet(worksheet_name)

    values = ws.get_all_values()
    if not values:
        return pd.DataFrame()

    headers = values[0]
    rows = values[1:]
    df = pd.DataFrame(rows, columns=headers)

    df = df.replace("", pd.NA).dropna(how="all")
    return df

raw = read_google_sheet_as_df(SPREADSHEET_ID, WORKSHEET_NAME, SERVICE_ACCOUNT_JSON_PATH)
raw.head(10)

,Timestamp,You are?,Workout date,Burnt >= 250 calories?
0,1/1/2026 15:30:34,Naveen Ganta,1/1/2026,Yes
1,1/1/2026 18:45:05,Eshwar,1/1/2026,Yes
2,1/1/2026 19:56:13,Srikar Gunisetty,1/1/2026,Yes
3,1/1/2026 21:27:05,Sandesh Ghanta,1/1/2026,Yes
4,1/1/2026 21:59:46,Surya Chaitanya,1/1/2026,Yes
5,1/1/2026 22:11:58,Vennela Chava,1/1/2026,Yes
6,1/1/2026 23:10:27,Pradyumna Ch.,1/1/2026,Yes
7,1/2/2026 19:57:22,Srivatsav Gunisetty,1/2/2026,Yes
8,1/2/2026 20:13:34,Pradyumna Ch.,1/2/2026,Yes
9,1/2/2026 22:04:31,Surya Chaitanya,1/2/2026,Yes


## <u>Cleaning and normalization</u>

Key cleaning steps:
- Parse Timestamp and Workout date into proper datetime types
- Normalize names (trim spaces, consistent casing)
- Standardize the 250+ calories flag into a boolean
- Deduplicate submissions so a person only gets credit once per workout date

Notes on data semantics:
- Qualifying workout day: a unique workout_date where burnt_250 is true
- Any workout day: a unique workout_date regardless of burnt_250 (used for consistency and streaks)

In [7]:
import pandas as pd
import numpy as np

def normalize_bool(x) -> bool:
    if pd.isna(x):
        return False
    s = str(x).strip().lower()
    return s in {"yes", "true", "1", "y", "t"}

def clean(
    df: pd.DataFrame,
    *,
    col_timestamp: str = "Timestamp",
    col_name: str = "You are?",
    col_wkdate: str = "Workout date",
    col_250: str = "Burnt >= 250 calories?",
    dedupe: bool = True,
) -> pd.DataFrame:
    df = df.copy()

    # if expected columns are missing
    expected = {col_timestamp, col_name, col_wkdate, col_250}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}. Found columns: {list(df.columns)}")

    # Rename for internal consistency
    df = df.rename(columns={
        col_timestamp: "timestamp",
        col_name: "name",
        col_wkdate: "workout_date",
        col_250: "burnt_250_raw",
    })

    # Parse datetimes
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # Workout date can be missing (NA) or a string; store as date
    wk = pd.to_datetime(df["workout_date"], errors="coerce")
    df["workout_date"] = wk.dt.date

    # Normalize name
    df["name_raw"] = df["name"]
    df["name"] = (
        df["name"]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Standardize boolean
    df["burnt_250"] = df["burnt_250_raw"].apply(normalize_bool)

    # Any workout means workout_date exists
    df["any_workout"] = df["workout_date"].notna()

    # Optional dedupe: one credit per person per workout day (earliest submission wins)
    if dedupe:
        df = df.sort_values(["name", "workout_date", "timestamp"])
        df = df.drop_duplicates(subset=["name", "workout_date"], keep="first")

    # Derived time columns (safe even when workout_date is missing)
    df["workout_dt"] = pd.to_datetime(df["workout_date"], errors="coerce")
    df["month"] = df["workout_dt"].dt.to_period("M").astype("string")
    df["dow"] = df["workout_dt"].dt.day_name()

    # Logging delay in days (timestamp_date - workout_date)
    df["timestamp_date"] = df["timestamp"].dt.date
    df["log_delay_days"] = (
        pd.to_datetime(df["timestamp_date"], errors="coerce")
        - pd.to_datetime(df["workout_date"], errors="coerce")
    ).dt.days

    # Day of month
    df["dom"] = df["workout_dt"].dt.day

    # Optional: flag suspicious negative delays (logged before the workout date)
    df["delay_flag"] = np.where(df["log_delay_days"].isna(), pd.NA,
                                np.where(df["log_delay_days"] < 0, "negative_delay", ""))

    return df

df = clean(raw)
df.head(10)

,timestamp,name,workout_date,burnt_250_raw,name_raw,burnt_250,any_workout,workout_dt,month,dow,timestamp_date,log_delay_days,dom,delay_flag
1,2026-01-01 18:45:05,Eshwar,2026-01-01,Yes,Eshwar,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,
12,2026-01-03 00:08:01,Eshwar,2026-01-02,No,Eshwar,False,True,2026-01-02,2026-01,Friday,2026-01-03,1,2,
0,2026-01-01 15:30:34,Naveen Ganta,2026-01-01,Yes,Naveen Ganta,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,
11,2026-01-02 22:56:32,Naveen Ganta,2026-01-02,Yes,Naveen Ganta,True,True,2026-01-02,2026-01,Friday,2026-01-02,0,2,
6,2026-01-01 23:10:27,Pradyumna Ch.,2026-01-01,Yes,Pradyumna Ch.,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,
8,2026-01-02 20:13:34,Pradyumna Ch.,2026-01-02,Yes,Pradyumna Ch.,True,True,2026-01-02,2026-01,Friday,2026-01-02,0,2,
3,2026-01-01 21:27:05,Sandesh Ghanta,2026-01-01,Yes,Sandesh Ghanta,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,
2,2026-01-01 19:56:13,Srikar Gunisetty,2026-01-01,Yes,Srikar Gunisetty,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,
7,2026-01-02 19:57:22,Srivatsav Gunisetty,2026-01-02,Yes,Srivatsav Gunisetty,True,True,2026-01-02,2026-01,Friday,2026-01-02,0,2,
4,2026-01-01 21:59:46,Surya Chaitanya,2026-01-01,Yes,Surya Chaitanya,True,True,2026-01-01,2026-01,Thursday,2026-01-01,0,1,


## Monthly Leaderboard

Winner definition:
- For a given month, a person is a winner if they have at least 16 unique workout dates in that month
- Only count dates where burnt_250 == True

We also compute:
- Total unique workout days (regardless of calories), to reflect consistency without filtering to 250+

In [9]:
WINNER_CUTOFF = 16

def monthly_summary(df: pd.DataFrame) -> pd.DataFrame:
    # Qualifying days (>=250) per person-month
    qualifying = (
        df[df["burnt_250"] & df["any_workout"]]
        .groupby(["month", "name"])["workout_date"]
        .nunique()
        .reset_index(name="qualifying_days_250")
    )

    # Any workout days per person-month
    any_days = (
        df[df["any_workout"]]
        .groupby(["month", "name"])["workout_date"]
        .nunique()
        .reset_index(name="workout_days_any")
    )

    out = any_days.merge(qualifying, on=["month", "name"], how="left")
    out["qualifying_days_250"] = out["qualifying_days_250"].fillna(0).astype(int)
    out["is_winner"] = out["qualifying_days_250"] >= WINNER_CUTOFF

    return out.sort_values(["month", "is_winner", "qualifying_days_250", "workout_days_any"], ascending=[True, False, False, False])

ms = monthly_summary(df)
ms.head(20)

,month,name,workout_days_any,qualifying_days_250,is_winner
1,2026-01,Naveen Ganta,2,2,False
2,2026-01,Pradyumna Ch.,2,2,False
6,2026-01,Surya Chaitanya,2,2,False
7,2026-01,Vennela Chava,2,2,False
0,2026-01,Eshwar,2,1,False
3,2026-01,Sandesh Ghanta,1,1,False
4,2026-01,Srikar Gunisetty,1,1,False
5,2026-01,Srivatsav Gunisetty,1,1,False


## Day-of-week trends

This section answers:
- Overall: which weekday is most preferred for workout by everyboady?
- Overall: which weekday is least preferred for workout by everyboady?
- Overall: list of folks who prefers weekends to workout vs who prefers weekdays?
- Per person: what weekday is most preferred and least preferred

Important note:
- Day-of-week analysis uses workout_date (not timestamp).

In [11]:
WEEKDAY_ORDER = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
WEEKEND_DAYS = {"Saturday", "Sunday"}

# Safety: ensure required columns exist
required_cols = {"name", "dow", "workout_date", "any_workout"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in df: {missing}. Found: {list(df.columns)}")

d = df[df["any_workout"]].copy()

# --- 1 & 2) Overall most/least preferred weekday ---
overall_dow = (
    d.groupby("dow")["workout_date"]
    .nunique()
    .reindex(WEEKDAY_ORDER)
    .fillna(0)
    .astype(int)
    .reset_index(name="unique_workout_days")
)

most_preferred_overall = overall_dow.loc[overall_dow["unique_workout_days"].idxmax()].to_dict()
least_preferred_overall = overall_dow.loc[overall_dow["unique_workout_days"].idxmin()].to_dict()

In [12]:
print("Overall weekday counts (unique workout dates across everyone):")
display(overall_dow)

Overall weekday counts (unique workout dates across everyone):


,dow,unique_workout_days
0,Monday,0
1,Tuesday,0
2,Wednesday,0
3,Thursday,1
4,Friday,1
5,Saturday,0
6,Sunday,0


In [13]:
print("\nOverall most preferred weekday:")
display(pd.DataFrame([most_preferred_overall]))


Overall most preferred weekday:


,dow,unique_workout_days
0,Thursday,1


In [14]:
print("\nOverall least preferred weekday:")
display(pd.DataFrame([least_preferred_overall]))


Overall least preferred weekday:


,dow,unique_workout_days
0,Monday,0


In [15]:
# --- 3) Weekend vs weekday preference per person ---
d = df[df["any_workout"]].copy()
d["is_weekend"] = d["dow"].isin(WEEKEND_DAYS).astype(bool)

weekend_weekday_pref = (
    d.groupby(["name", "is_weekend"])["workout_date"]
    .nunique()
    .reset_index(name="unique_days")
)

pref = weekend_weekday_pref.pivot(index="name", columns="is_weekend", values="unique_days")

# Always force BOTH columns to exist, and in a known order
pref = pref.reindex(columns=[False, True], fill_value=0).fillna(0)

# Rename to friendly column names
pref.columns = ["weekday_days", "weekend_days"]

pref["preference"] = np.select(
    [
        pref["weekend_days"] > pref["weekday_days"],
        pref["weekend_days"] < pref["weekday_days"],
    ],
    ["Weekend", "Weekday"],
    default="Tie"
)

pref_out = (
    pref.reset_index()
    .sort_values(["preference", "weekend_days", "weekday_days"], ascending=[True, False, False])
)

In [16]:
print("\nWeekend vs Weekday preference per person:")
display(pref_out)


Weekend vs Weekday preference per person:


,name,weekday_days,weekend_days,preference
0,Eshwar,2,0,Weekday
1,Naveen Ganta,2,0,Weekday
2,Pradyumna Ch.,2,0,Weekday
6,Surya Chaitanya,2,0,Weekday
7,Vennela Chava,2,0,Weekday
3,Sandesh Ghanta,1,0,Weekday
4,Srikar Gunisetty,1,0,Weekday
5,Srivatsav Gunisetty,1,0,Weekday


In [17]:
print("\nPeople who prefer weekends:")
display(pref_out[pref_out["preference"] == "Weekend"])


People who prefer weekends:


,name,weekday_days,weekend_days,preference


In [18]:
print("\nPeople who prefer weekdays:")
display(pref_out[pref_out["preference"] == "Weekday"])


People who prefer weekdays:


,name,weekday_days,weekend_days,preference
0,Eshwar,2,0,Weekday
1,Naveen Ganta,2,0,Weekday
2,Pradyumna Ch.,2,0,Weekday
6,Surya Chaitanya,2,0,Weekday
7,Vennela Chava,2,0,Weekday
3,Sandesh Ghanta,1,0,Weekday
4,Srikar Gunisetty,1,0,Weekday
5,Srivatsav Gunisetty,1,0,Weekday


In [19]:
print("\nPeople tied (same number of weekend and weekday workouts):")
display(pref_out[pref_out["preference"] == "Tie"])


People tied (same number of weekend and weekday workouts):


,name,weekday_days,weekend_days,preference


In [20]:
# --- 4) Per person most/least preferred weekday (including zeros) ---
per_person_dow = (
    d.groupby(["name", "dow"])["workout_date"]
    .nunique()
    .reset_index(name="unique_days")
)

# Ensure every person has all 7 weekdays represented (fill missing with 0)
full_index = pd.MultiIndex.from_product(
    [per_person_dow["name"].unique(), WEEKDAY_ORDER],
    names=["name", "dow"]
)

per_person_dow_full = (
    per_person_dow
    .set_index(["name", "dow"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# Most preferred weekday per person (ties: returns all tied days)
max_days = per_person_dow_full.groupby("name")["unique_days"].transform("max")
most_pref = per_person_dow_full[per_person_dow_full["unique_days"] == max_days].copy()
most_pref = most_pref.rename(columns={"dow": "most_preferred_day", "unique_days": "most_preferred_days"})

# Least preferred weekday per person (ties: returns all tied days, often zeros)
min_days = per_person_dow_full.groupby("name")["unique_days"].transform("min")
least_pref = per_person_dow_full[per_person_dow_full["unique_days"] == min_days].copy()
least_pref = least_pref.rename(columns={"dow": "least_preferred_day", "unique_days": "least_preferred_days"})

In [21]:
# Find each person's max workout count across weekdays
per_person_dow_full["max_days"] = per_person_dow_full.groupby("name")["unique_days"].transform("max")

# Keep only the weekdays tied for max (their preferred days)
preferred = per_person_dow_full[per_person_dow_full["unique_days"] == per_person_dow_full["max_days"]].copy()

# Collect preferred days into a Python list per person
preferred_days_list = (
    preferred.sort_values(["name", "dow"], key=lambda s: s.map({d:i for i,d in enumerate(WEEKDAY_ORDER)}))
    .groupby("name")["dow"]
    .apply(list)
    .reset_index(name="preferred_workout_days")
)

preferred_days_list

,name,preferred_workout_days
0,Eshwar,"[Thursday, Friday]"
1,Naveen Ganta,"[Thursday, Friday]"
2,Pradyumna Ch.,"[Thursday, Friday]"
3,Sandesh Ghanta,[Thursday]
4,Srikar Gunisetty,[Thursday]
5,Srivatsav Gunisetty,[Friday]
6,Surya Chaitanya,"[Thursday, Friday]"
7,Vennela Chava,"[Thursday, Friday]"


In [22]:
# Optional: single-row summary per person by joining tied days into comma-separated strings
most_joined = (
    most_pref.groupby("name")
    .agg(
        most_preferred_day=("most_preferred_day", lambda x: ", ".join(x)),
        most_preferred_days=("most_preferred_days", "max"),
    )
)

least_joined = (
    least_pref.groupby("name")
    .agg(
        least_preferred_day=("least_preferred_day", lambda x: ", ".join(x)),
        least_preferred_days=("least_preferred_days", "min"),
    )
)

dow_summary = (
    most_joined
    .merge(least_joined, left_index=True, right_index=True, how="left")
    .merge(pref[["weekday_days", "weekend_days", "preference"]], left_index=True, right_index=True, how="left")
    .reset_index()
    .rename(columns={"index": "name"})
    .sort_values("name")
)

print("\nPer-person day-of-week summary (single row per person):")
display(dow_summary)


Per-person day-of-week summary (single row per person):


,name,most_preferred_day,most_preferred_days,least_preferred_day,least_preferred_days,weekday_days,weekend_days,preference
0,Eshwar,"Thursday, Friday",1,"Monday, Tuesday, Wednesday, Saturday, Sunday",0,2,0,Weekday
1,Naveen Ganta,"Thursday, Friday",1,"Monday, Tuesday, Wednesday, Saturday, Sunday",0,2,0,Weekday
2,Pradyumna Ch.,"Thursday, Friday",1,"Monday, Tuesday, Wednesday, Saturday, Sunday",0,2,0,Weekday
3,Sandesh Ghanta,Thursday,1,"Monday, Tuesday, Wednesday, Friday, Saturday, ...",0,1,0,Weekday
4,Srikar Gunisetty,Thursday,1,"Monday, Tuesday, Wednesday, Friday, Saturday, ...",0,1,0,Weekday
5,Srivatsav Gunisetty,Friday,1,"Monday, Tuesday, Wednesday, Thursday, Saturday...",0,1,0,Weekday
6,Surya Chaitanya,"Thursday, Friday",1,"Monday, Tuesday, Wednesday, Saturday, Sunday",0,2,0,Weekday
7,Vennela Chava,"Thursday, Friday",1,"Monday, Tuesday, Wednesday, Saturday, Sunday",0,2,0,Weekday


## Logging delay analysis (timestamp vs workout date)

This section answers:
- Who logs workouts the latest (higher delay)?
- Median and mean delay per person per month
- Percent logged same-day vs 1+ days late

Interpretation:
- log_delay_days = 0 means logged on the same calendar day
- positive values mean logged late
- negative values usually indicate data entry issues (logged before the workout date) and should be reviewed

In [24]:
def logging_delay_summary(df: pd.DataFrame) -> pd.DataFrame:
    d = df[df["any_workout"]].copy()

    agg = (
        d.groupby(["month","name"])["log_delay_days"]
        .agg(
            submissions="count",
            mean_delay="mean",
            median_delay="median",
            p90_delay=lambda x: np.nanpercentile(x, 90) if len(x) else np.nan,
            same_day_rate=lambda x: float(np.mean(x == 0)) if len(x) else np.nan,
            late_rate=lambda x: float(np.mean(x > 0)) if len(x) else np.nan,
            negative_delay_rate=lambda x: float(np.mean(x < 0)) if len(x) else np.nan,
        )
        .reset_index()
    )

    # "Laziness" proxy score: prioritize median, then p90
    agg["lazy_score"] = agg["median_delay"].fillna(0) + 0.25 * agg["p90_delay"].fillna(0)
    return agg.sort_values(["month", "lazy_score"], ascending=[True, False])

ld = logging_delay_summary(df)
ld.head(20)

,month,name,submissions,mean_delay,median_delay,p90_delay,same_day_rate,late_rate,negative_delay_rate,lazy_score
0,2026-01,Eshwar,2,0.5,0.5,0.9,0.5,0.5,0.0,0.725
1,2026-01,Naveen Ganta,2,0.0,0.0,0.0,1.0,0.0,0.0,0.000
2,2026-01,Pradyumna Ch.,2,0.0,0.0,0.0,1.0,0.0,0.0,0.000
3,2026-01,Sandesh Ghanta,1,0.0,0.0,0.0,1.0,0.0,0.0,0.000
4,2026-01,Srikar Gunisetty,1,0.0,0.0,0.0,1.0,0.0,0.0,0.000
5,2026-01,Srivatsav Gunisetty,1,0.0,0.0,0.0,1.0,0.0,0.0,0.000
6,2026-01,Surya Chaitanya,2,0.0,0.0,0.0,1.0,0.0,0.0,0.000
7,2026-01,Vennela Chava,2,0.0,0.0,0.0,1.0,0.0,0.0,0.000


## Longest Streak

- Gives a list of top 3 candidates who maintained the longest streak of continuous work out dates in a given month.
- Order them by streak length and give me top 3 candidates

In [26]:
# -----------------------------
# Longest streaks per person-month (two columns)
# 1) longest_streak_any: any workout day
# 2) longest_streak_250: only days where burnt_250 == True
# Plus: Top 3 per month by each streak metric
# -----------------------------

def longest_streak_from_dates(dates) -> int:
    """
    dates: iterable of date-like objects (python date or datetime)
    Returns length of longest consecutive-day streak.
    """
    ds = (
        pd.to_datetime(pd.Series(list(dates)), errors="coerce")
        .dropna()
        .dt.date
        .unique()
    )
    if len(ds) == 0:
        return 0

    ords = np.sort([d.toordinal() for d in ds])

    best = cur = 1
    for i in range(1, len(ords)):
        if ords[i] - ords[i - 1] == 1:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return int(best)

# Compute streaks per person-month
streaks = (
    df[df["any_workout"] & df["workout_date"].notna()]
    .groupby(["month", "name"])
    .apply(lambda g: pd.Series({
        "longest_streak_any": longest_streak_from_dates(g["workout_date"]),
        "longest_streak_250": longest_streak_from_dates(g.loc[g["burnt_250"], "workout_date"]),
    }))
    .reset_index()
)

display(streaks.sort_values(["month", "longest_streak_any"], ascending=[True, False]).head(20))

# Top 3 per month by ANY-workout streak
top3_any_streak = (
    streaks.sort_values(["month", "longest_streak_any", "name"], ascending=[True, False, True])
           .groupby("month")
           .head(3)
           .reset_index(drop=True)
)
# Top 3 per month by >=250 streak
top3_250_streak = (
    streaks.sort_values(["month", "longest_streak_250", "name"], ascending=[True, False, True])
           .groupby("month")
           .head(3)
           .reset_index(drop=True)
)

C:\Users\srika\AppData\Local\Temp\ipykernel_3604\1881950717.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,month,name,longest_streak_any,longest_streak_250
0,2026-01,Eshwar,2,1
1,2026-01,Naveen Ganta,2,2
2,2026-01,Pradyumna Ch.,2,2
6,2026-01,Surya Chaitanya,2,2
7,2026-01,Vennela Chava,2,2
3,2026-01,Sandesh Ghanta,1,1
4,2026-01,Srikar Gunisetty,1,1
5,2026-01,Srivatsav Gunisetty,1,1


In [27]:
print("Top 3 per month by ANY-workout streak:")
display(top3_any_streak)

Top 3 per month by ANY-workout streak:


,month,name,longest_streak_any,longest_streak_250
0,2026-01,Eshwar,2,1
1,2026-01,Naveen Ganta,2,2
2,2026-01,Pradyumna Ch.,2,2


In [28]:
print("Top 3 per month by >=250-calorie streak:")
display(top3_250_streak)

Top 3 per month by >=250-calorie streak:


,month,name,longest_streak_any,longest_streak_250
0,2026-01,Naveen Ganta,2,2
1,2026-01,Pradyumna Ch.,2,2
2,2026-01,Surya Chaitanya,2,2


## <u> Fastest Winner </u>

In [30]:
# -----------------------------
# Fastest winner
# Definition: for each person-month, find the date when they achieved
# their 16th qualifying workout day (burnt_250 == True).
# The earlier that 16th date occurs in the month, the "faster" the winner.
# -----------------------------

WINNER_CUTOFF = 16  # adjust if needed

# Qualifying records only
q = df[df["any_workout"] & df["burnt_250"] & df["workout_date"].notna()].copy()

# Get sorted unique qualifying workout dates per person-month
q_dates = (
    q.groupby(["month", "name"])["workout_date"]
     .apply(lambda s: sorted(set(s)))
     .reset_index(name="qualifying_dates_sorted")
)

# Keep only those who have >= 16 qualifying days
q_dates["qualifying_days_250"] = q_dates["qualifying_dates_sorted"].apply(len)
winners = q_dates[q_dates["qualifying_days_250"] >= WINNER_CUTOFF].copy()

# The "clinch date" is the 16th qualifying date
winners["clinch_date_16th"] = winners["qualifying_dates_sorted"].apply(lambda xs: xs[WINNER_CUTOFF - 1])

# Day of month they clinched (lower is faster)
winners["clinch_day_of_month"] = pd.to_datetime(winners["clinch_date_16th"]).dt.day

# Optional: how many days from month start (0-based)
month_start = pd.to_datetime(winners["month"] + "-01")
winners["days_from_month_start"] = (pd.to_datetime(winners["clinch_date_16th"]) - month_start).dt.days

# Rank winners per month by how fast they clinched
winners["fastest_rank_in_month"] = (
    winners.groupby("month")["clinch_day_of_month"]
           .rank(method="dense", ascending=True)
           .astype(int)
)

fastest_winners_by_month = (
    winners.sort_values(["month", "clinch_day_of_month", "name"], ascending=[True, True, True])
           .reset_index(drop=True)
)

In [31]:
print("Fastest winners by month (earliest clinch of 16 qualifying days):")
display(
    fastest_winners_by_month[[
        "month", "name", "qualifying_days_250",
        "clinch_date_16th", "clinch_day_of_month", "fastest_rank_in_month"
    ]]
)

Fastest winners by month (earliest clinch of 16 qualifying days):


,month,name,qualifying_days_250,clinch_date_16th,clinch_day_of_month,fastest_rank_in_month


In [32]:
# Top 1 fastest winner per month
top1_fastest_each_month = (
    fastest_winners_by_month.groupby("month").head(1).reset_index(drop=True)
)

print("Top 1 fastest winner per month:")
display(top1_fastest_each_month[[
    "month", "name", "clinch_date_16th", "clinch_day_of_month", "qualifying_days_250"
]])

Top 1 fastest winner per month:


,month,name,clinch_date_16th,clinch_day_of_month,qualifying_days_250


In [33]:
# Overall fastest winners across all months (top 10)
overall_fastest = (
    fastest_winners_by_month.sort_values(["clinch_day_of_month", "month", "name"], ascending=[True, True, True])
                            .head(10)
                            .reset_index(drop=True)
)

print("Overall fastest winners across all months (top 10):")
display(overall_fastest[[
    "month", "name", "clinch_date_16th", "clinch_day_of_month", "qualifying_days_250"
]])

Overall fastest winners across all months (top 10):


,month,name,clinch_date_16th,clinch_day_of_month,qualifying_days_250


## Front-loading vs cramming (workouts early vs late in the month)

This section answers:
- Who tends to complete workouts earlier in the month vs later?
- Quantify behavior with a simple score

Approach:
- For each person-month, compute the average day-of-month for workout_date (lower means earlier)
- Compute share of workouts in:
  - first 10 days
  - last 10 days (or last third, depending on preference)

Interpretation:
- Higher "cram_score" means more workouts concentrated at the end of the month

In [35]:

# Safety check
required_cols = {"name", "month", "workout_date", "any_workout"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

d = df[df["any_workout"] & df["workout_date"].notna()].copy()

# Convert workout_date to datetime for safety
d["workout_dt"] = pd.to_datetime(d["workout_date"])

# Day of month
d["dom"] = d["workout_dt"].dt.day

# Days in month (needed to compute last-10-days correctly)
d["month_start"] = pd.to_datetime(d["month"] + "-01")
d["days_in_month"] = d["month_start"].dt.days_in_month

# Aggregate per person-month
frontload_cram = (
    d.groupby(["month", "name"])
    .apply(lambda g: pd.Series({
        "workout_days_any": g["workout_date"].nunique(),
        # Average day-of-month (lower = earlier workouts)
        "avg_day_of_month": g["dom"].mean(),
        # Share of workouts in first 10 days
        "share_first_10_days": (g["dom"] <= 10).mean(),
        # Share of workouts in last 10 days
        "share_last_10_days": (g["dom"] >= (g["days_in_month"].iloc[0] - 9)).mean(),
    }))
    .reset_index()
)

# Cram score: positive => more late-month workouts
frontload_cram["cram_score"] = (
    frontload_cram["share_last_10_days"]
    - frontload_cram["share_first_10_days"]
)

# Optional categorical interpretation
frontload_cram["behavior"] = np.select(
    [
        frontload_cram["cram_score"] > 0.2,
        frontload_cram["cram_score"] < -0.2,
    ],
    [
        "Crammer (late-month heavy)",
        "Front-loader (early-month heavy)",
    ],
    default="Balanced"
)

# Sort for inspection (most extreme behaviors first)
frontload_cram_sorted = (
    frontload_cram
    .sort_values(["month", "cram_score"], ascending=[True, False])
    .reset_index(drop=True)
)

display(frontload_cram_sorted)


C:\Users\srika\AppData\Local\Temp\ipykernel_3604\4238966746.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,month,name,workout_days_any,avg_day_of_month,share_first_10_days,share_last_10_days,cram_score,behavior
0,2026-01,Eshwar,2.0,1.5,1.0,0.0,-1.0,Front-loader (early-month heavy)
1,2026-01,Naveen Ganta,2.0,1.5,1.0,0.0,-1.0,Front-loader (early-month heavy)
2,2026-01,Pradyumna Ch.,2.0,1.5,1.0,0.0,-1.0,Front-loader (early-month heavy)
3,2026-01,Sandesh Ghanta,1.0,1.0,1.0,0.0,-1.0,Front-loader (early-month heavy)
4,2026-01,Srikar Gunisetty,1.0,1.0,1.0,0.0,-1.0,Front-loader (early-month heavy)
5,2026-01,Srivatsav Gunisetty,1.0,2.0,1.0,0.0,-1.0,Front-loader (early-month heavy)
6,2026-01,Surya Chaitanya,2.0,1.5,1.0,0.0,-1.0,Front-loader (early-month heavy)
7,2026-01,Vennela Chava,2.0,1.5,1.0,0.0,-1.0,Front-loader (early-month heavy)


## Barely missed and Less Intensive workouts

Barely missed:
- People with 14 or 15 qualifying days (>=250) in the month

Less Intensive but consistent workouts
- People who met 16 day cut off but with < 250 calories

In [37]:
WINNER_CUTOFF = 16
BARELY_LOW, BARELY_HIGH = WINNER_CUTOFF - 2, WINNER_CUTOFF - 1

# Build (or reuse) the monthly leaderboard table: ms
# If you already have ms, you can skip this block.
qualifying = (
    df[df["any_workout"] & df["burnt_250"]]
    .groupby(["month", "name"])["workout_date"]
    .nunique()
    .reset_index(name="qualifying_days_250")
)

any_days = (
    df[df["any_workout"]]
    .groupby(["month", "name"])["workout_date"]
    .nunique()
    .reset_index(name="workout_days_any")
)

ms = (
    any_days.merge(qualifying, on=["month", "name"], how="left")
           .fillna({"qualifying_days_250": 0})
)
ms["qualifying_days_250"] = ms["qualifying_days_250"].astype(int)
ms["is_winner"] = ms["qualifying_days_250"] >= WINNER_CUTOFF

# 1) Barely missed
barely_missed = (
    ms[ms["qualifying_days_250"].between(BARELY_LOW, BARELY_HIGH)]
    .sort_values(["month", "qualifying_days_250", "workout_days_any", "name"], ascending=[True, False, False, True])
    .reset_index(drop=True)
)

In [38]:
print("Barely missed (14–15 qualifying days >=250):")
display(barely_missed[["month", "name", "qualifying_days_250", "workout_days_any"]])

Barely missed (14–15 qualifying days >=250):


,month,name,qualifying_days_250,workout_days_any


In [39]:
# 2) Less intensive but consistent
less_intensive_consistent = (
    ms[(ms["workout_days_any"] >= WINNER_CUTOFF) & (ms["qualifying_days_250"] < WINNER_CUTOFF)]
    .assign(non_qualifying_days=lambda x: x["workout_days_any"] - x["qualifying_days_250"])
    .sort_values(["month", "workout_days_any", "qualifying_days_250", "name"], ascending=[True, False, False, True])
    .reset_index(drop=True)
)

print("Less intensive but consistent (>=16 workout days, but <16 qualifying >=250 days):")
display(less_intensive_consistent[[
    "month", "name", "workout_days_any", "qualifying_days_250", "non_qualifying_days"
]])

Less intensive but consistent (>=16 workout days, but <16 qualifying >=250 days):


,month,name,workout_days_any,qualifying_days_250,non_qualifying_days


## Month-over-month trends

This section provides:
- Total participants per month
- Total workouts per month
- Average qualifying days among participants
- Winner count per month
  
These help you understand engagement growth and seasonality.

In [41]:
# Safety check
required_cols = {"month", "name", "workout_date", "any_workout", "burnt_250"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

d = df[df["any_workout"] & df["workout_date"].notna()].copy()

# ---- Core aggregates ----

# 1) Total participants per month
participants = (
    d.groupby("month")["name"]
     .nunique()
     .reset_index(name="participants")
)

# 2) Total workouts per month (unique workout dates across everyone)
total_workouts = (
    d.groupby("month")["workout_date"]
     .nunique()
     .reset_index(name="total_workout_days")
)

# 3) Qualifying days per person-month
qualifying_days = (
    d[d["burnt_250"]]
    .groupby(["month", "name"])["workout_date"]
    .nunique()
    .reset_index(name="qualifying_days_250")
)

# 4) Average qualifying days among participants
avg_qualifying = (
    qualifying_days
    .groupby("month")["qualifying_days_250"]
    .mean()
    .reset_index(name="avg_qualifying_days_250")
)

# 5) Winner count per month
winner_count = (
    qualifying_days
    .assign(is_winner=lambda x: x["qualifying_days_250"] >= WINNER_CUTOFF)
    .groupby("month")["is_winner"]
    .sum()
    .reset_index(name="winner_count")
)

# ---- Final month-over-month table ----
mom_trends = (
    participants
    .merge(total_workouts, on="month", how="left")
    .merge(avg_qualifying, on="month", how="left")
    .merge(winner_count, on="month", how="left")
    .sort_values("month")
    .reset_index(drop=True)
)

display(mom_trends)

,month,participants,total_workout_days,avg_qualifying_days_250,winner_count
0,2026-01,8,2,1.5,0


## Visualizations

In [43]:
def set_plot_style():
    """
    Global plot styling to be called once near the top of the notebook.
    """
    sns.set_theme(style="darkgrid", palette="muted")
    mpl.rcParams["figure.dpi"] = 150
    plt.rcParams["font.family"] = "Consolas"

def style_axes(ax, title=None, xlabel=None, ylabel=None):
    """
    Enforces title/label font requirements on a given Axes.
    - Title: bold, size 12
    - X/Y labels: bold, size 8
    """
    if title is not None:
        ax.set_title(title, fontweight="bold", fontsize=12)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontweight="bold", fontsize=8)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontweight="bold", fontsize=8)
    ax.tick_params(labelsize=8)
    return ax

# Call once
set_plot_style()

In [44]:
# 1) Monthly qualifying leaderboard (bar)
def plot_01_monthly_qualifying_leaderboard(df, month, top_n=20, cutoff=16):
    ms = _monthly_leaderboard(df, cutoff=cutoff)
    g = ms[ms["month"] == month].sort_values("qualifying_days_250", ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=g, x="name", y="qualifying_days_250", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Qualifying Days (>=250) Leaderboard - {month}", "Name", "Qualifying days")
    plt.tight_layout()
    return ax

In [45]:
# 2) Monthly consistency leaderboard (bar)
def plot_02_monthly_consistency_leaderboard(df, month, top_n=20, cutoff=16):
    ms = _monthly_leaderboard(df, cutoff=cutoff)
    g = ms[ms["month"] == month].sort_values("workout_days_any", ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=g, x="name", y="workout_days_any", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Workout Days (Any) Leaderboard - {month}", "Name", "Workout days")
    plt.tight_layout()
    return ax

# 3) Winners highlight (qualifying bar + cutoff line)
def plot_03_winners_highlight(df, month, cutoff=16):
    ms = _monthly_leaderboard(df, cutoff=cutoff)
    g = ms[ms["month"] == month].sort_values("qualifying_days_250", ascending=False)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=g, x="name", y="qualifying_days_250", ax=ax)
    ax.axhline(cutoff, linestyle="--", linewidth=1)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Winners Highlight (Cutoff={cutoff}) - {month}", "Name", "Qualifying days")
    plt.tight_layout()
    return ax

# 4) Barely missed (14–15) bar
def plot_04_barely_missed(df, month, low=14, high=15, cutoff=16):
    ms = _monthly_leaderboard(df, cutoff=cutoff)
    g = ms[(ms["month"] == month) & (ms["qualifying_days_250"].between(low, high))].sort_values("qualifying_days_250", ascending=False)
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.barplot(data=g, x="name", y="qualifying_days_250", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Barely Missed (Qualifying {low}-{high}) - {month}", "Name", "Qualifying days")
    plt.tight_layout()
    return ax

# 5) Less-intensive but consistent (two metrics per person; scatter overlay)
def plot_05_less_intensive_consistent(df, month, cutoff=16, top_n=30):
    ms = _monthly_leaderboard(df, cutoff=cutoff)
    g = ms[(ms["month"] == month) & (ms["workout_days_any"] >= cutoff) & (ms["qualifying_days_250"] < cutoff)]
    g = g.sort_values("workout_days_any", ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=g, x="name", y="workout_days_any", ax=ax)
    sns.scatterplot(data=g, x="name", y="qualifying_days_250", ax=ax, legend=False)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Less-Intensive but Consistent (>= {cutoff} workouts) - {month}", "Name", "Days (bar=any, dot=qualifying)")
    plt.tight_layout()
    return ax

# 6) Participants over time (line)
def plot_06_participants_over_time(df, cutoff=16):
    mom = _mom_trends(df, cutoff=cutoff)
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.lineplot(data=mom, x="month", y="participants", marker="o", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, "Participants Over Time", "Month", "Participants")
    plt.tight_layout()
    return ax

# 7) Winner count over time (line)
def plot_07_winners_over_time(df, cutoff=16):
    mom = _mom_trends(df, cutoff=cutoff)
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.lineplot(data=mom, x="month", y="winner_count", marker="o", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, "Winner Count Over Time", "Month", "Winners")
    plt.tight_layout()
    return ax

# 8) Total workouts over time (line)
def plot_08_total_workouts_over_time(df, cutoff=16):
    mom = _mom_trends(df, cutoff=cutoff)
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.lineplot(data=mom, x="month", y="total_workout_days", marker="o", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, "Total Workout Days Over Time", "Month", "Unique workout days")
    plt.tight_layout()
    return ax

# 9) Avg qualifying days over time (line)
def plot_09_avg_qualifying_over_time(df, cutoff=16):
    mom = _mom_trends(df, cutoff=cutoff)
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.lineplot(data=mom, x="month", y="avg_qualifying_days_250", marker="o", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, "Average Qualifying Days (>=250) Over Time", "Month", "Avg qualifying days")
    plt.tight_layout()
    return ax

# 10) MoM deltas (bar) for participants
def plot_10_mom_delta_participants(df, cutoff=16):
    mom = _mom_trends(df, cutoff=cutoff).copy()
    mom["delta_participants"] = mom["participants"].diff()
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.barplot(data=mom, x="month", y="delta_participants", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, "MoM Change in Participants", "Month", "Delta participants")
    plt.tight_layout()
    return ax

# 11) Overall weekday distribution (bar)
def plot_11_overall_weekday_distribution(df):
    d = df[df["any_workout"] & df["workout_date"].notna()].copy()
    overall = (
        d.groupby("dow")["workout_date"].nunique()
        .reindex(WEEKDAY_ORDER).fillna(0).astype(int).reset_index(name="unique_workout_days")
    )
    fig, ax = plt.subplots(figsize=(7, 3))
    sns.barplot(data=overall, x="dow", y="unique_workout_days", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, "Overall Workouts by Weekday", "Weekday", "Unique workout days")
    plt.tight_layout()
    return ax

# 12) Per-person weekday counts (stacked bar using pivot)
def plot_12_person_weekday_stacked(df, month=None, top_n=20):
    d = df[df["any_workout"] & df["workout_date"].notna()].copy()
    if month is not None:
        d = d[d["month"] == month]
    per = d.groupby(["name","dow"])["workout_date"].nunique().reset_index(name="days")
    pivot = per.pivot(index="name", columns="dow", values="days").reindex(columns=WEEKDAY_ORDER).fillna(0)
    pivot["total"] = pivot.sum(axis=1)
    pivot = pivot.sort_values("total", ascending=False).head(top_n).drop(columns="total")

    fig, ax = plt.subplots(figsize=(10, 5))
    bottom = np.zeros(len(pivot))
    x = np.arange(len(pivot.index))
    for day in WEEKDAY_ORDER:
        vals = pivot[day].values
        ax.bar(x, vals, bottom=bottom, label=day)
        bottom += vals
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=45, ha="right")
    style_axes(ax, f"Per-Person Workouts by Weekday{' - '+month if month else ''}", "Name", "Unique workout days")
    ax.legend(fontsize=7, ncols=4)
    plt.tight_layout()
    return ax

# 13) Weekday vs weekend preference (bar of weekend share)
def plot_13_weekend_share_per_person(df, month=None):
    d = df[df["any_workout"] & df["workout_date"].notna()].copy()
    if month is not None:
        d = d[d["month"] == month]
    d["is_weekend"] = d["dow"].isin(WEEKEND_DAYS).astype(bool)
    agg = d.groupby(["name","is_weekend"])["workout_date"].nunique().reset_index(name="days")
    p = agg.pivot(index="name", columns="is_weekend", values="days").reindex(columns=[False, True], fill_value=0).fillna(0)
    p.columns = ["weekday_days","weekend_days"]
    p["weekend_share"] = p["weekend_days"] / (p["weekday_days"] + p["weekend_days"]).replace(0, np.nan)
    out = p.reset_index().sort_values("weekend_share", ascending=False)

    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=out, x="name", y="weekend_share", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Weekend Share by Person{' - '+month if month else ''}", "Name", "Weekend share")
    plt.tight_layout()
    return ax

# 14) Heatmap: person vs weekday
def plot_14_heatmap_person_weekday(df, month=None, top_n=30):
    d = df[df["any_workout"] & df["workout_date"].notna()].copy()
    if month is not None:
        d = d[d["month"] == month]
    per = d.groupby(["name","dow"])["workout_date"].nunique().reset_index(name="days")
    pivot = per.pivot(index="name", columns="dow", values="days").reindex(columns=WEEKDAY_ORDER).fillna(0)
    pivot["total"] = pivot.sum(axis=1)
    pivot = pivot.sort_values("total", ascending=False).head(top_n).drop(columns="total")

    fig, ax = plt.subplots(figsize=(8, 0.35*len(pivot)+2))
    sns.heatmap(pivot, annot=False, cmap=None, ax=ax)
    style_axes(ax, f"Heatmap: Workouts by Weekday{' - '+month if month else ''}", "Weekday", "Name")
    plt.tight_layout()
    return ax

# 15) Logging delay distribution (hist)
def plot_15_delay_hist(df, month=None, bins=20):
    d = df[df["any_workout"] & df["workout_date"].notna() & df["log_delay_days"].notna()].copy()
    if month is not None:
        d = d[d["month"] == month]
    fig, ax = plt.subplots(figsize=(7, 3))
    sns.histplot(d["log_delay_days"], bins=bins, ax=ax)
    style_axes(ax, f"Logging Delay Distribution{' - '+month if month else ''}", "Delay days (timestamp - workout_date)", "Count")
    plt.tight_layout()
    return ax

# 16) Median delay per person (bar)
def plot_16_median_delay_per_person(df, month=None, top_n=30):
    ld = _logging_delay(df)
    if month is not None:
        ld = ld[ld["month"] == month]
    out = ld.sort_values("median_delay", ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=out, x="name", y="median_delay", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Median Logging Delay by Person{' - '+month if month else ''}", "Name", "Median delay (days)")
    plt.tight_layout()
    return ax

# 17) Late logging rate per person (bar)
def plot_17_late_rate_per_person(df, month=None, top_n=30):
    ld = _logging_delay(df)
    if month is not None:
        ld = ld[ld["month"] == month]
    out = ld.sort_values("late_rate", ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=out, x="name", y="late_rate", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Late Logging Rate by Person{' - '+month if month else ''}", "Name", "Late rate")
    plt.tight_layout()
    return ax

# 18) Delay vs qualifying days (scatter)
def plot_18_delay_vs_qualifying(df, month=None, cutoff=16):
    ms = _monthly_leaderboard(df, cutoff=cutoff)
    ld = _logging_delay(df)
    if month is not None:
        ms = ms[ms["month"] == month]
        ld = ld[ld["month"] == month]
    merged = ms.merge(ld, on=["month","name"], how="left")
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.scatterplot(data=merged, x="median_delay", y="qualifying_days_250", hue="is_winner", ax=ax)
    style_axes(ax, f"Median Delay vs Qualifying Days{' - '+month if month else ''}", "Median delay (days)", "Qualifying days")
    plt.tight_layout()
    return ax

# 19) Cram score per person (bar)
def plot_19_cram_score_per_person(df, month=None, top_n=30):
    fc = _frontload_cram(df)
    if month is not None:
        fc = fc[fc["month"] == month]
    out = fc.sort_values("cram_score", ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=out, x="name", y="cram_score", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Cram Score by Person{' - '+month if month else ''}", "Name", "Cram score (last10 - first10)")
    plt.tight_layout()
    return ax

# 20) Avg day-of-month per person (bar)
def plot_20_avg_dom_per_person(df, month=None, top_n=30):
    fc = _frontload_cram(df)
    if month is not None:
        fc = fc[fc["month"] == month]
    out = fc.sort_values("avg_day_of_month", ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=out, x="name", y="avg_day_of_month", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Average Day-of-Month of Workouts{' - '+month if month else ''}", "Name", "Avg day-of-month")
    plt.tight_layout()
    return ax

# 21) First10 vs last10 shares (side-by-side bars)
def plot_21_first10_last10_shares(df, month=None, top_n=30):
    fc = _frontload_cram(df)
    if month is not None:
        fc = fc[fc["month"] == month]
    out = fc.sort_values("workout_days_any", ascending=False).head(top_n).copy()
    melt = out.melt(id_vars=["month","name"], value_vars=["share_first_10_days","share_last_10_days"],
                    var_name="window", value_name="share")
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=melt, x="name", y="share", hue="window", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"First 10 vs Last 10 Share{' - '+month if month else ''}", "Name", "Share")
    ax.legend(fontsize=7)
    plt.tight_layout()
    return ax

# 22) Cram score vs qualifying days (scatter)
def plot_22_cram_vs_qualifying(df, month=None, cutoff=16):
    fc = _frontload_cram(df)
    ms = _monthly_leaderboard(df, cutoff=cutoff)
    merged = ms.merge(fc, on=["month","name"], how="left")
    if month is not None:
        merged = merged[merged["month"] == month]
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.scatterplot(data=merged, x="cram_score", y="qualifying_days_250", hue="is_winner", ax=ax)
    style_axes(ax, f"Cram Score vs Qualifying Days{' - '+month if month else ''}", "Cram score", "Qualifying days")
    plt.tight_layout()
    return ax

# 23) Top 3 streaks per month (any) as bar (one month at a time)
def plot_23_top3_streak_any(df, month):
    st = _streaks(df)
    g = st[st["month"] == month].sort_values("longest_streak_any", ascending=False).head(3)
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=g, x="name", y="longest_streak_any", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Top 3 Longest Streaks (Any) - {month}", "Name", "Streak length (days)")
    plt.tight_layout()
    return ax

# 24) Streak comparison (grouped bars: any vs 250)
def plot_24_streak_comparison(df, month=None, top_n=20):
    st = _streaks(df)
    if month is not None:
        st = st[st["month"] == month]
    st = st.sort_values("longest_streak_any", ascending=False).head(top_n)
    melt = st.melt(id_vars=["month","name"], value_vars=["longest_streak_any","longest_streak_250"],
                   var_name="type", value_name="streak_days")
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=melt, x="name", y="streak_days", hue="type", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Streak Comparison{' - '+month if month else ''}", "Name", "Longest streak (days)")
    ax.legend(fontsize=7)
    plt.tight_layout()
    return ax

# 25) Streak vs consistency (scatter)
def plot_25_streak_vs_consistency(df, month=None, cutoff=16):
    st = _streaks(df)
    ms = _monthly_leaderboard(df, cutoff=cutoff)
    merged = ms.merge(st, on=["month","name"], how="left")
    if month is not None:
        merged = merged[merged["month"] == month]
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.scatterplot(data=merged, x="workout_days_any", y="longest_streak_any", hue="is_winner", ax=ax)
    style_axes(ax, f"Consistency vs Longest Streak{' - '+month if month else ''}", "Workout days (any)", "Longest streak (any)")
    plt.tight_layout()
    return ax

# 26) Clinch day leaderboard (fastest winner) bar
def plot_26_clinch_day_leaderboard(df, month, cutoff=16):
    fw = _fastest_winner(df, cutoff=cutoff)
    g = fw[fw["month"] == month].sort_values("clinch_day_of_month", ascending=True)
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.barplot(data=g, x="name", y="clinch_day_of_month", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_axes(ax, f"Fastest Winners (Clinch Day) - {month}", "Name", "Day of month clinched")
    plt.tight_layout()
    return ax

# 27) Calendar pace curve: cumulative qualifying days vs day-of-month (top_k people)
def plot_27_cumulative_qualifying_pace(df, month, top_k=5):
    d = df[df["any_workout"] & df["burnt_250"] & df["workout_date"].notna()].copy()
    d = d[d["month"] == month].copy()
    d["workout_dt"] = pd.to_datetime(d["workout_date"])
    d["dom"] = d["workout_dt"].dt.day

    # choose top_k by total qualifying days
    totals = d.groupby("name")["workout_date"].nunique().sort_values(ascending=False).head(top_k).index.tolist()
    d = d[d["name"].isin(totals)].copy()

    # compute cumulative per person
    d = d.sort_values(["name","dom"])
    cum = (
        d.groupby(["name","dom"])["workout_date"].nunique()
        .groupby(level=0).cumsum()
        .reset_index(name="cum_qualifying_days")
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.lineplot(data=cum, x="dom", y="cum_qualifying_days", hue="name", marker="o", ax=ax)
    style_axes(ax, f"Cumulative Qualifying Pace - {month}", "Day of month", "Cumulative qualifying days")
    ax.legend(fontsize=7)
    plt.tight_layout()
    return ax

# 28) Workout calendar heatmap per person (monthly) - binary any-workout
def plot_28_person_month_calendar_heatmap(df, name, month):
    """
    Simple calendar-like heatmap:
    - rows: weekday (Mon..Sun)
    - cols: week index within month (1..6)
    - cell: 1 if worked out that date, else 0
    """
    d = df[df["any_workout"] & df["workout_date"].notna()].copy()
    d = d[(d["name"] == name) & (d["month"] == month)].copy()
    if d.empty:
        raise ValueError(f"No data for name={name}, month={month}")

    # Build a full date range for the month
    month_start = pd.to_datetime(month + "-01")
    dim = month_start.days_in_month
    all_dates = pd.date_range(month_start, periods=dim, freq="D")

    cal = pd.DataFrame({"date": all_dates})
    cal["dow"] = cal["date"].dt.day_name()
    cal["dom"] = cal["date"].dt.day

    # week index within month (1-based), aligned by actual calendar weeks
    # Use ISO week number difference from first day
    cal["week_index"] = ((cal["date"].dt.dayofyear - month_start.dayofyear) + month_start.dayofweek) // 7 + 1

    worked = set(pd.to_datetime(pd.Series(d["workout_date"])).dt.date)
    cal["worked_out"] = cal["date"].dt.date.apply(lambda x: 1 if x in worked else 0)

    pivot = (
        cal.pivot(index="dow", columns="week_index", values="worked_out")
        .reindex(WEEKDAY_ORDER)
        .fillna(0)
        .astype(int)
    )

    fig, ax = plt.subplots(figsize=(6, 3))
    sns.heatmap(pivot, annot=True, fmt="d", cbar=False, ax=ax)
    style_axes(ax, f"Workout Calendar (Any) - {name} - {month}", "Week of month", "Weekday")
    plt.tight_layout()
    return ax

# 29) Month activity heatmap: day-of-month vs month (overall intensity)
def plot_29_month_dayofmonth_heatmap(df):
    """
    Heatmap of activity volume:
    - rows: month
    - cols: day-of-month (1..31)
    - values: number of workouts logged that day (unique people count)
    """
    d = df[df["any_workout"] & df["workout_date"].notna()].copy()
    d["workout_dt"] = pd.to_datetime(d["workout_date"])
    d["dom"] = d["workout_dt"].dt.day

    # intensity = number of unique people who worked out on that date
    intensity = (
        d.groupby(["month","dom"])["name"].nunique()
        .reset_index(name="active_people")
    )

    pivot = intensity.pivot(index="month", columns="dom", values="active_people").fillna(0).sort_index()

    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(pivot, annot=False, ax=ax)
    style_axes(ax, "Monthly Activity Heatmap (People Active per Day-of-Month)", "Day of month", "Month")
    plt.tight_layout()
    return ax

In [50]:
from IPython.display import display

def _show_ax(ax):
    """Force-render a seaborn/matplotlib plot in notebooks."""
    fig = ax.figure
    display(fig)
    plt.show(fig)  # prevents duplicate rendering later

def plot_monthly_report(df, month, *, cutoff=16, top_n=20, calendar_name=None):
    # Leaderboards
    _show_ax(plot_01_monthly_qualifying_leaderboard(df, month, top_n=top_n, cutoff=cutoff))
    _show_ax(plot_02_monthly_consistency_leaderboard(df, month, top_n=top_n, cutoff=cutoff))
    _show_ax(plot_03_winners_highlight(df, month, cutoff=cutoff))

    # Barely missed + less-intensive
    _show_ax(plot_04_barely_missed(df, month, low=cutoff-2, high=cutoff-1, cutoff=cutoff))
    _show_ax(plot_05_less_intensive_consistent(df, month, cutoff=cutoff, top_n=top_n))

    # Day-of-week
    _show_ax(plot_11_overall_weekday_distribution(df[df["month"] == month].copy()))
    _show_ax(plot_12_person_weekday_stacked(df, month=month, top_n=min(top_n, 15)))
    _show_ax(plot_13_weekend_share_per_person(df, month=month))
    _show_ax(plot_14_heatmap_person_weekday(df, month=month, top_n=min(top_n, 25)))

    # Logging delay
    _show_ax(plot_15_delay_hist(df, month=month))
    _show_ax(plot_16_median_delay_per_person(df, month=month, top_n=min(top_n, 25)))
    _show_ax(plot_17_late_rate_per_person(df, month=month, top_n=min(top_n, 25)))
    _show_ax(plot_18_delay_vs_qualifying(df, month=month, cutoff=cutoff))

    # Front-load vs cram
    _show_ax(plot_19_cram_score_per_person(df, month=month, top_n=min(top_n, 25)))
    _show_ax(plot_20_avg_dom_per_person(df, month=month, top_n=min(top_n, 25)))
    _show_ax(plot_21_first10_last10_shares(df, month=month, top_n=min(top_n, 15)))
    _show_ax(plot_22_cram_vs_qualifying(df, month=month, cutoff=cutoff))

    # Streaks + fastest winner + pace
    _show_ax(plot_23_top3_streak_any(df, month))
    _show_ax(plot_24_streak_comparison(df, month=month, top_n=min(top_n, 15)))
    _show_ax(plot_25_streak_vs_consistency(df, month=month, cutoff=cutoff))
    _show_ax(plot_26_clinch_day_leaderboard(df, month=month, cutoff=cutoff))
    _show_ax(plot_27_cumulative_qualifying_pace(df, month=month, top_k=5))

    # Optional calendar heatmap
    if calendar_name:
        _show_ax(plot_28_person_month_calendar_heatmap(df, name=calendar_name, month=month))


def plot_overall_trends_report(df, *, cutoff=16):
    _show_ax(plot_06_participants_over_time(df, cutoff=cutoff))
    _show_ax(plot_07_winners_over_time(df, cutoff=cutoff))
    _show_ax(plot_08_total_workouts_over_time(df, cutoff=cutoff))
    _show_ax(plot_09_avg_qualifying_over_time(df, cutoff=cutoff))
    _show_ax(plot_10_mom_delta_participants(df, cutoff=cutoff))
    _show_ax(plot_29_month_dayofmonth_heatmap(df))

## Run the plots

Use these helpers to render the full set of plots. If you only want a specific month, call `plot_monthly_report(df, month)`.


In [48]:
# Sanity checks
assert "month" in df.columns and "name" in df.columns, "Expected standardized columns (month, name, ...)."
months = sorted([m for m in df["month"].dropna().unique().tolist()])
print("Rows:", len(df), "| Months:", months[:8], ("..." if len(months)>8 else ""))

# Overall / MoM trend plots
plot_overall_trends_report(df, cutoff=16)

# Monthly report plots (uncomment to render all months)
# for m in months:
#     print("\n=== ", m, " ===")
#     plot_monthly_report(df, m, cutoff=16, top_n=20)

# Example: render the latest month only
if months:
    latest = months[-1]
    print("\nRendering latest month:", latest)
    plot_monthly_report(df, latest, cutoff=16, top_n=20)

Rows: 13 | Months: ['2026-01'] 


NameError: name '_mom_trends' is not defined